# 🎨 PhotoEnhance — AI Photo Upscaler
**Безкоштовне покращення якості фото за допомогою Real-ESRGAN**

---

| Параметр | Значення |
|---|---|
| Модель | Real-ESRGAN |
| GPU | Google Colab T4 (16GB) |
| Масштаб | x2, x4, x8 |
| Формати | JPG, PNG, WEBP |
| Вартість | $0 |

> ⚠️ **Переконайся, що увімкнено GPU:** Runtime → Change runtime type → T4 GPU

In [ ]:
# @title ⚙️ Cell 1: Setup — Встановлення Real-ESRGAN та залежностей { display-mode: "form" }
# @markdown Запускай тільки **один раз** на початку сесії.

print('📦 Встановлення залежностей...')

!pip install -q basicsr facexlib gfpgan
!pip install -q realesrgan
!pip install -q ipywidgets Pillow opencv-python-headless

# ─── Фікс сумісності basicsr з новим torchvision ─────────────────────
# torchvision >= 0.16 видалив functional_tensor — патчимо вручну
import sys
from types import ModuleType
import torchvision.transforms.functional as _tvf

_compat = ModuleType('torchvision.transforms.functional_tensor')
_compat.rgb_to_grayscale = _tvf.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = _compat
print('🔧 torchvision compatibility patch applied')

# Клонуємо репозиторій для отримання скрипту inference
import os
if not os.path.exists('/content/Real-ESRGAN'):
    !git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN
    %cd /content/Real-ESRGAN
    !pip install -q -r requirements.txt
    !python setup.py develop -q
else:
    %cd /content/Real-ESRGAN

print('✅ Встановлення завершено!')
print('👉 Перейди до Cell 2 для перевірки GPU')


In [ ]:
# @title 🔍 Cell 2: GPU Check — Перевірка наявності GPU { display-mode: "form" }

import subprocess
import torch

print('=' * 50)
print('🖥️  ІНФОРМАЦІЯ ПРО GPU')
print('=' * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU знайдено: {gpu_name}')
    print(f'💾 VRAM: {gpu_memory:.1f} GB')
    print(f'🔧 CUDA версія: {torch.version.cuda}')
    
    # nvidia-smi output
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,temperature.gpu',
                             '--format=csv,noheader,nounits'], capture_output=True, text=True)
    if result.returncode == 0:
        parts = result.stdout.strip().split(', ')
        if len(parts) >= 4:
            print(f'🌡️  Температура: {parts[3]}°C')
            print(f'📊 Вільна пам\'ять: {int(parts[2])//1024:.1f} GB / {int(parts[1])//1024:.1f} GB')
    
    print('\n✅ Все готово для обробки фото!')
else:
    print('❌ GPU не знайдено!')
    print('👉 Перейди: Runtime → Change runtime type → T4 GPU → Save')
    print('   Потім перезапусти всі клітинки знову.')

print('=' * 50)

In [ ]:
# @title 📁 Cell 3: Google Drive — Підключення для зберігання результатів { display-mode: "form" }
# @markdown Результати будуть збережені у папці `PhotoEnhance_Results` на твоєму Drive.
# @markdown
# @markdown 💡 **Потрібен особистий @gmail.com** — шкільні/корпоративні акаунти можуть не мати доступу до Colab.

from google.colab import drive
import os

TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

try:
    print('🔗 Підключення Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    OUTPUT_DIR = '/content/drive/MyDrive/PhotoEnhance_Results'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    DRIVE_AVAILABLE = True

    print(f'✅ Google Drive підключено!')
    print(f'📂 Результати: {OUTPUT_DIR}')

except Exception as e:
    print(f'⚠️  Drive не підключено: {e}')
    print('   Файли будуть доступні тільки для скачування у Cell 6.')
    OUTPUT_DIR = TEMP_DIR
    DRIVE_AVAILABLE = False

print(f'📂 Тимчасова папка: {TEMP_DIR}')
print('\n👉 Перейди до Cell 4 для завантаження фото!')


In [ ]:
# @title 🖼️ Cell 4: Інтерфейс — Завантаження фото та вибір налаштувань { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import io
from PIL import Image
import os

# ─── Стилі ───────────────────────────────────────────────────────────
display(HTML("""
<style>
  .widget-label { font-weight: bold; font-size: 14px; }
  .pe-title { font-family: sans-serif; color: #1a1a2e; margin-bottom: 10px; }
  .pe-info  { font-family: monospace; background: #f0f4ff; padding: 8px;
               border-radius: 6px; border-left: 4px solid #4a90e2; }
</style>
"""))

# ─── Заголовок ───────────────────────────────────────────────────────
title_html = widgets.HTML("""
<div class="pe-title">
  <h2>🎨 PhotoEnhance MVP</h2>
  <p style="color:#666;">Безкоштовне AI покращення якості фото — powered by Real-ESRGAN</p>
</div>
""")

# ─── Кнопка завантаження ─────────────────────────────────────────────
upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp',
    multiple=False,
    description='📸 Вибрати фото',
    layout=widgets.Layout(width='200px', height='40px')
)

# ─── Вибір масштабу ──────────────────────────────────────────────────
scale_options = widgets.ToggleButtons(
    options=[('× 2  (швидше)', 2), ('× 4  (оптимально)', 4), ('× 8  (max якість)', 8)],
    value=4,
    description='Масштаб:',
    style={'button_width': '160px'}
)

# ─── Вибір моделі ────────────────────────────────────────────────────
model_options = widgets.Dropdown(
    options=[
        ('🖼️  Загальні фото (реальні сцени)', 'RealESRGAN_x4plus'),
        ('🎌 Аніме / Ілюстрації', 'RealESRGAN_x4plus_anime_6B'),
        ('📷 Старі / деградовані фото', 'ESRGAN_SRx4_DF2KOST'),
        ('⚡ Швидкий режим (x2)', 'RealESRGAN_x2plus'),
    ],
    value='RealESRGAN_x4plus',
    description='Модель:',
    layout=widgets.Layout(width='400px')
)

# ─── Прев'ю завантаженого фото ───────────────────────────────────────
preview_output = widgets.Output()
info_label = widgets.HTML('<div class="pe-info">📁 Фото не вибрано</div>')

def on_upload_change(change):
    if upload_btn.value:
        for fname, fdata in upload_btn.value.items():
            img = Image.open(io.BytesIO(fdata['content']))
            size_kb = len(fdata['content']) / 1024
            info_label.value = f"""
            <div class="pe-info">
              ✅ <b>{fname}</b><br>
              📐 Розмір: {img.size[0]}×{img.size[1]} px | {size_kb:.0f} KB<br>
              🎨 Режим: {img.mode}
            </div>
            """
            with preview_output:
                clear_output(wait=True)
                img_thumb = img.copy()
                img_thumb.thumbnail((300, 300))
                import matplotlib.pyplot as plt
                fig, ax = plt.subplots(figsize=(4, 4))
                ax.imshow(img_thumb)
                ax.set_title('Оригінал (прев\'ю)', fontsize=10)
                ax.axis('off')
                plt.tight_layout()
                plt.show()

upload_btn.observe(on_upload_change, names='value')

# ─── Кнопка запуску обробки ────────────────────────────────────────
process_btn = widgets.Button(
    description='🚀 Покращити фото!',
    button_style='success',
    layout=widgets.Layout(width='200px', height='45px'),
    style={'font_weight': 'bold'}
)

process_status = widgets.Output()

def on_process_click(b):
    with process_status:
        clear_output(wait=True)
        if not upload_btn.value:
            print('⚠️ Спочатку завантаж фото!')
            return
        print('⏳ Починаємо обробку... Запускай Cell 5!')

process_btn.on_click(on_process_click)

# ─── Компонування UI ─────────────────────────────────────────────────
ui = widgets.VBox([
    title_html,
    widgets.HBox([upload_btn]),
    info_label,
    preview_output,
    widgets.HTML('<hr>'),
    model_options,
    scale_options,
    widgets.HTML('<hr>'),
    process_btn,
    process_status,
])

display(ui)
print('\n👉 Після вибору фото — запусти Cell 5 для обробки!')

In [ ]:
# @title ⚡ Cell 5: Обробка фото — AI Enhancement { display-mode: "form" }
# @markdown Запускай цю клітинку після вибору фото в Cell 4.

import os
import io
import time
import urllib.request
import sys
from types import ModuleType

# ─── Фікс сумісності basicsr з новим torchvision ─────────────────────
# Потрібно застосувати ДО імпорту realesrgan/basicsr
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    import torchvision.transforms.functional as _tvf
    _compat = ModuleType('torchvision.transforms.functional_tensor')
    _compat.rgb_to_grayscale = _tvf.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _compat

import numpy as np
import cv2
import torch
from PIL import Image
from IPython.display import clear_output

# ─── Конфігурація моделей ────────────────────────────────────────────
MODEL_URLS = {
    'RealESRGAN_x4plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        4
    ),
    'RealESRGAN_x4plus_anime_6B': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth',
        4
    ),
    'RealESRGAN_x2plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
        2
    ),
    'ESRGAN_SRx4_DF2KOST': (
        'https://github.com/xinntao/ESRGAN/releases/download/v0.1.1/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth',
        4
    ),
}

MODELS_DIR = '/content/models'
os.makedirs(MODELS_DIR, exist_ok=True)

def download_model(model_name: str) -> str:
    """Завантажує модель якщо не існує."""
    url, _ = MODEL_URLS[model_name]
    model_path = os.path.join(MODELS_DIR, f'{model_name}.pth')
    if not os.path.exists(model_path):
        print(f'⬇️  Завантаження моделі {model_name} (~67MB)...')
        urllib.request.urlretrieve(url, model_path)
        print(f'✅ Модель завантажено: {model_path}')
    else:
        print(f'✅ Модель вже є: {model_name}')
    return model_path

def build_upsampler(model_name: str, outscale: int):
    """Ініціалізує RealESRGANer для вибраної моделі."""
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet

    model_path = download_model(model_name)
    _, model_scale = MODEL_URLS[model_name]

    # Параметри архітектури залежать від моделі
    if 'anime_6B' in model_name:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_block=6, num_grow_ch=32, scale=model_scale)
    else:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_block=23, num_grow_ch=32, scale=model_scale)

    upsampler = RealESRGANer(
        scale=model_scale,
        model_path=model_path,
        model=model,
        tile=400,        # розбиваємо на тайли — уникаємо OOM
        tile_pad=10,
        pre_pad=0,
        half=torch.cuda.is_available(),  # fp16 на GPU, fp32 на CPU
    )
    return upsampler


def enhance_image(input_path: str, output_path: str,
                  model_name: str, outscale: int) -> dict:
    """Основна функція покращення якості зображення."""
    # Читаємо зображення
    img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'Не вдалося прочитати файл: {input_path}')

    h_orig, w_orig = img.shape[:2]
    print(f'📐 Оригінальний розмір: {w_orig}×{h_orig} px')
    print(f'🤖 Модель: {model_name} | Масштаб: ×{outscale}')

    # Запускаємо upsampler
    upsampler = build_upsampler(model_name, outscale)

    t_start = time.time()
    print('🔄 Обробка...')
    output_img, _ = upsampler.enhance(img, outscale=outscale)
    elapsed = time.time() - t_start

    h_out, w_out = output_img.shape[:2]
    print(f'✅ Готово за {elapsed:.1f} сек')
    print(f'📐 Результуючий розмір: {w_out}×{h_out} px')

    # Зберігаємо результат
    cv2.imwrite(output_path, output_img)
    print(f'💾 Збережено: {output_path}')

    return {
        'orig_size': (w_orig, h_orig),
        'out_size': (w_out, h_out),
        'elapsed': elapsed,
        'output_path': output_path
    }


# ─── Основний блок обробки ───────────────────────────────────────────
if not upload_btn.value:
    print('⚠️  Спочатку завантаж фото в Cell 4!')
else:
    selected_model = model_options.value
    selected_scale = scale_options.value

    for fname, fdata in upload_btn.value.items():
        # Зберігаємо вхідний файл
        input_path = f'/content/photoenhance_temp/{fname}'
        with open(input_path, 'wb') as f:
            f.write(fdata['content'])

        # Шлях до результату
        base, ext = os.path.splitext(fname)
        result_name = f'{base}_enhanced_x{selected_scale}.png'
        local_output = f'/content/photoenhance_temp/{result_name}'
        drive_output = f'{OUTPUT_DIR}/{result_name}'

        print('=' * 55)
        print(f'🎨 PhotoEnhance — Обробка файлу: {fname}')
        print('=' * 55)

        stats = enhance_image(input_path, local_output, selected_model, selected_scale)

        # Копіюємо на Drive (якщо підключено)
        import shutil
        if DRIVE_AVAILABLE and drive_output != local_output:
            shutil.copy2(local_output, drive_output)
            print(f'☁️  Збережено на Drive: {drive_output}')
        else:
            print('💾 Drive не підключено — скачай файл у Cell 6')

        # Зберігаємо шляхи для Cell 6
        ORIGINAL_PATH = input_path
        ENHANCED_PATH = local_output
        PROCESS_STATS = stats

    print('\n✅ Обробку завершено! Запусти Cell 6 для перегляду результату.')


In [ ]:
# @title 👁️ Cell 6: Before / After — Порівняння результатів { display-mode: "form" }
# @markdown Порівняння оригіналу та покращеного фото + скачування.

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from google.colab import files
import numpy as np

try:
    orig_img  = Image.open(ORIGINAL_PATH).convert('RGB')
    enh_img   = Image.open(ENHANCED_PATH).convert('RGB')
    stats     = PROCESS_STATS
except NameError:
    print('⚠️  Спочатку виконай Cell 5 (обробку фото)!')
    raise

# ─── Побудова графіку ────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 9), facecolor='#1a1a2e')
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.03)

# Ліворуч — оригінал
ax1 = fig.add_subplot(gs[0])
ax1.imshow(orig_img)
ax1.set_title(
    f'ОРИГІНАЛ\n{stats["orig_size"][0]}×{stats["orig_size"][1]} px',
    color='white', fontsize=14, pad=12
)
ax1.axis('off')
ax1.text(0.02, 0.02, '◀ BEFORE', transform=ax1.transAxes,
         color='white', fontsize=12, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#e74c3c', alpha=0.85))

# Праворуч — покращене
ax2 = fig.add_subplot(gs[1])
ax2.imshow(enh_img)
ax2.set_title(
    f'ПОКРАЩЕНО (×{selected_scale})\n{stats["out_size"][0]}×{stats["out_size"][1]} px',
    color='white', fontsize=14, pad=12
)
ax2.axis('off')
ax2.text(0.02, 0.02, 'AFTER ▶', transform=ax2.transAxes,
         color='white', fontsize=12, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#2ecc71', alpha=0.85))

# Заголовок
fig.suptitle(
    f'🎨 PhotoEnhance — Real-ESRGAN  |  Час обробки: {stats["elapsed"]:.1f}s',
    color='white', fontsize=16, fontweight='bold', y=1.01
)

# Зберігаємо порівняльне зображення
comparison_path = '/content/photoenhance_temp/comparison.png'
plt.savefig(comparison_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

# ─── Статистика ──────────────────────────────────────────────────────
scale_factor = selected_scale
pixel_increase = (stats['out_size'][0] * stats['out_size'][1]) / \
                 (stats['orig_size'][0] * stats['orig_size'][1])

print('\n' + '=' * 55)
print('📊 СТАТИСТИКА ОБРОБКИ')
print('=' * 55)
print(f'  📥 Оригінал:    {stats["orig_size"][0]:>5} × {stats["orig_size"][1]} px')
print(f'  📤 Результат:   {stats["out_size"][0]:>5} × {stats["out_size"][1]} px')
print(f'  🔍 Масштаб:     ×{scale_factor}')
print(f'  📈 Пікселів:    ×{pixel_increase:.1f} більше')
print(f'  ⏱️  Час:          {stats["elapsed"]:.1f} сек')
print(f'  🤖 Модель:      {selected_model}')
print('=' * 55)

# ─── Скачування ──────────────────────────────────────────────────────
print('\n⬇️  Завантаження покращеного фото...')
files.download(ENHANCED_PATH)

print('\n⬇️  Завантаження порівняльного зображення...')
files.download(comparison_path)

print('\n✅ Готово! Файли збережено і на Google Drive.')